# 08end_to_end_demo 노트북 목표
1. 저장된 프로젝트 제안 모델을 실제 리뷰 입력에 적용한다.
2. 리뷰 텍스트에서 KcBERT 임베딩과 메타데이터를 생성한다.
3. 이벤트 리뷰 확률, 이벤트 여부, 후보 2 별점 정제 가중치를 출력한다.
4. 예시 가게 리뷰 여러 개를 입력해 기존 평균 별점과 정제 후 별점을 비교한다.

이 노트북은 모델을 새로 학습하지 않고, 01~07번에서 만든 `outputs/final_proposed_model.joblib`을 실제 입력에 적용하는 end-to-end 데모이다. 또한 성능 기준 최종 선택 모델이 아니라, 프로젝트에서 제안한 KcBERT PCA + metadata 구조의 실제 적용 흐름을 보여준다.

## 1. 라이브러리 로드

- 저장된 모델 bundle을 불러오고, KcBERT 임베딩과 입력 feature를 만들기 위한 패키지를 임포트한다.

In [15]:
from pathlib import Path
import re

import numpy as np
import pandas as pd
import joblib

import emoji
from soynlp.normalizer import repeat_normalize

import torch
from transformers import AutoTokenizer, AutoModel

## 2. 저장 모델과 KcBERT 로드

- 06번에서 저장한 프로젝트 제안 모델 `final_proposed_model.joblib`만 사용한다.
- KcBERT는 02번과 동일하게 `beomi/kcbert-base`를 특징 추출기로 사용한다.
- 저장 bundle의 `feature_cols`를 그대로 사용해 학습 당시 입력 순서와 맞춘다.

In [16]:
MODEL_BUNDLE_PATH = Path('outputs/final_proposed_model.joblib')
KCBERT_MODEL_NAME = 'beomi/kcbert-base'

if not MODEL_BUNDLE_PATH.exists():
    raise FileNotFoundError(
        f'제안 모델 bundle이 없습니다: {MODEL_BUNDLE_PATH}. 01~07번을 먼저 실행하세요.'
    )

model_bundle = joblib.load(MODEL_BUNDLE_PATH)
event_model = model_bundle['model']
feature_cols = model_bundle.get('feature_cols')
threshold = float(model_bundle.get('best_threshold', model_bundle.get('default_threshold', 0.5)))

if feature_cols is None:
    raise ValueError('저장 모델 bundle에 feature_cols가 없습니다.')

expected_embedding_cols = [f'kcbert_{i}' for i in range(768)]
expected_meta_cols = ['text_length', 'emoji_count', 'photo_count']
expected_feature_cols = expected_embedding_cols + expected_meta_cols
missing_required_cols = [col for col in expected_feature_cols if col not in feature_cols]
if missing_required_cols:
    raise ValueError(f'제안 모델 feature_cols에 필요한 컬럼이 없습니다: {missing_required_cols[:10]}')

assert 'rating' not in feature_cols, 'rating은 별점 정제 대상이므로 모델 입력에 포함되면 안 됩니다.'

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
tokenizer = AutoTokenizer.from_pretrained(KCBERT_MODEL_NAME)
kcbert_model = AutoModel.from_pretrained(KCBERT_MODEL_NAME).to(device)
kcbert_model.eval()

print('제안 모델:', model_bundle.get('model_name'))
print('feature set:', model_bundle.get('feature_set'))
print('feature 수:', len(feature_cols))
print('threshold:', round(threshold, 4))
print('KcBERT 실행 환경:', device)

제안 모델: proposed_hybrid_mlp_04
feature set: kcbert_pca+metadata
feature 수: 771
threshold: 0.5009
KcBERT 실행 환경: cuda


## 3. 리뷰 전처리와 feature 생성 함수

- 01번과 같은 방식으로 리뷰 텍스트를 정제한다.
- 정제 텍스트는 KcBERT 입력으로 사용하고, 원문 리뷰에서 `text_length`, `emoji_count`, `photo_count`를 만든다.
- KcBERT `[CLS]` 벡터 768차원과 메타데이터 3개를 합쳐 모델 입력 feature를 만든다.

In [17]:
pattern = re.compile(r'[^ .,?!/@$%~％·∼()\x00-\x7F ㄱ-ㅣ가-힣]+')
url_pattern = re.compile(
    r'https?:\/\/(www\.)?[-a-zA-Z0-9@:%._\+~#=]{1,256}'
    r'\.[a-zA-Z0-9()]{1,6}\b'
    r'([-a-zA-Z0-9()@:%_\+.~#?&//=]*)'
)


def clean_review_text(text):
    text = str(text)
    text = url_pattern.sub('', text)
    text = emoji.replace_emoji(text, replace='')
    text = pattern.sub(' ', text)
    text = text.strip()
    text = repeat_normalize(text, num_repeats=2)
    return text


def count_emoji(text):
    return len([char for char in str(text) if char in emoji.EMOJI_DATA])


def normalize_photo_count(photo_count):
    if photo_count is None or pd.isna(photo_count):
        return 0
    return int(photo_count)


def embed_texts(cleaned_texts, batch_size=16):
    cls_embeddings = []
    with torch.no_grad():
        for start in range(0, len(cleaned_texts), batch_size):
            batch_text = cleaned_texts[start:start + batch_size]
            inputs = tokenizer(
                batch_text,
                return_tensors='pt',
                truncation=True,
                max_length=128,
                padding='max_length',
            ).to(device)
            outputs = kcbert_model(**inputs)
            cls_vector = outputs.last_hidden_state[:, 0, :].cpu().numpy()
            cls_embeddings.append(cls_vector)

    return np.vstack(cls_embeddings)


def make_feature_frame(review_texts, photo_counts):
    if isinstance(review_texts, str):
        review_texts = [review_texts]
    if np.isscalar(photo_counts) or photo_counts is None:
        photo_counts = [photo_counts] * len(review_texts)

    if len(review_texts) != len(photo_counts):
        raise ValueError('review_texts와 photo_counts의 길이가 같아야 합니다.')

    rows = []
    cleaned_texts = []
    for review_text, photo_count in zip(review_texts, photo_counts):
        raw_text = str(review_text)
        cleaned_text = clean_review_text(raw_text)
        cleaned_texts.append(cleaned_text)
        rows.append({
            'review_text': raw_text,
            'cleaned_review_text': cleaned_text,
            'text_length': len(raw_text),
            'emoji_count': count_emoji(raw_text),
            'photo_count': normalize_photo_count(photo_count),
        })

    embeddings = embed_texts(cleaned_texts)
    embedding_df = pd.DataFrame(embeddings, columns=expected_embedding_cols)
    info_df = pd.DataFrame(rows)
    feature_df = pd.concat([embedding_df, info_df[expected_meta_cols]], axis=1)

    missing_model_cols = [col for col in feature_cols if col not in feature_df.columns]
    if missing_model_cols:
        raise KeyError(f'모델 입력 feature를 만들 수 없습니다: {missing_model_cols[:10]}')

    return feature_df[feature_cols], info_df

## 4. 단일 리뷰 이벤트 판별

- 리뷰 1개와 사진 개수를 입력하면 이벤트 리뷰 확률을 출력한다.
- 이벤트 확률이 threshold 이상이면 이벤트 리뷰로 판단한다.
- 후보 2 가중치는 `1 - event_prob(이벤트 확률)`이며, 이벤트 가능성이 높을수록 별점 반영 비중이 낮아진다.

In [18]:
DEMO_REVIEW_TEXT = '리뷰 이벤트 참여합니다. 콜라 서비스 부탁드려요! 음식도 맛있었어요 😊'
DEMO_PHOTO_COUNT = 1

single_feature, single_info = make_feature_frame(DEMO_REVIEW_TEXT, DEMO_PHOTO_COUNT)
single_event_prob = float(event_model.predict_proba(single_feature)[:, 1][0])
single_event_pred = int(single_event_prob >= threshold)
single_candidate_2_weight = float(np.clip(1 - single_event_prob, 0, 1))

single_result = pd.DataFrame([{
    '원문 리뷰': single_info.loc[0, 'review_text'],
    '정제 리뷰': single_info.loc[0, 'cleaned_review_text'],
    '사진 수': single_info.loc[0, 'photo_count'],
    '텍스트 길이': single_info.loc[0, 'text_length'],
    '이모지 수': single_info.loc[0, 'emoji_count'],
    'threshold': threshold,
    '이벤트 확률': single_event_prob,
    '이벤트 예측': '이벤트 리뷰' if single_event_pred == 1 else '일반 리뷰',
    '후보 2 가중치': single_candidate_2_weight,
}]).round({
    'threshold': 4,
    '이벤트 확률': 4,
    '후보 2 가중치': 4,
})

assert 0 <= single_event_prob <= 1
assert np.isclose(single_candidate_2_weight, 1 - single_event_prob)

display(single_result)

,원문 리뷰,정제 리뷰,사진 수,텍스트 길이,이모지 수,threshold,이벤트 확률,이벤트 예측,후보 2 가중치
0,리뷰 이벤트 참여합니다. 콜라 서비스 부탁드려요! 음식도 맛있었어요 😊,리뷰 이벤트 참여합니다. 콜라 서비스 부탁드려요! 음식도 맛있었어요,1,39,1,0.5009,0.628,이벤트 리뷰,0.372


## 5. 가게 단위 별점 정제 데모

- 예시 가게의 여러 리뷰를 한 번에 입력한다.
- 각 리뷰의 이벤트 확률을 계산하고, 후보 2 가중 평균으로 가게 별점을 정제한다.
- 정제 공식은 `sum(rating * (1 - event_prob)) / sum(1 - event_prob)`이다.

In [19]:
demo_store_reviews = pd.DataFrame([
    {'review_text': '리뷰 이벤트 참여합니다 콜라 부탁드려요', 'rating': 5.0, 'photo_count': 1},
    {'review_text': '양 많고 맛있어요 다음에 또 시킬게요', 'rating': 5.0, 'photo_count': 0},
    {'review_text': '음식이 조금 식어서 왔어요 그래도 괜찮았습니다', 'rating': 3.0, 'photo_count': 0},
    {'review_text': '포토리뷰 약속했어요 서비스 감사합니다', 'rating': 5.0, 'photo_count': 2},
    {'review_text': '배달도 빠르고 포장도 깔끔했어요', 'rating': 4.0, 'photo_count': 0},
])

store_feature, store_info = make_feature_frame(
    demo_store_reviews['review_text'].tolist(),
    demo_store_reviews['photo_count'].tolist(),
)
store_event_prob = event_model.predict_proba(store_feature)[:, 1]
store_event_pred = (store_event_prob >= threshold).astype(int)
store_candidate_2_weight = np.clip(1 - store_event_prob, 0, 1)

demo_store_result = demo_store_reviews.copy()
demo_store_result['cleaned_review_text'] = store_info['cleaned_review_text']
demo_store_result['event_prob'] = store_event_prob
demo_store_result['event_pred'] = store_event_pred
demo_store_result['event_pred_name'] = np.where(store_event_pred == 1, '이벤트 리뷰', '일반 리뷰')
demo_store_result['candidate_2_weight'] = store_candidate_2_weight
demo_store_result['candidate_2_weighted_rating'] = demo_store_result['rating'] * demo_store_result['candidate_2_weight']

original_mean_rating = float(demo_store_result['rating'].mean())
weight_sum = float(demo_store_result['candidate_2_weight'].sum())
refined_rating = (
    original_mean_rating
    if weight_sum <= 0
    else float(demo_store_result['candidate_2_weighted_rating'].sum() / weight_sum)
)
rating_delta = refined_rating - original_mean_rating

store_summary = pd.DataFrame([{
    '리뷰 수': len(demo_store_result),
    '이벤트 예측 수': int(demo_store_result['event_pred'].sum()),
    '기존 평균 별점': original_mean_rating,
    '정제 후 별점': refined_rating,
    '별점 변화량': rating_delta,
}]).round({
    '기존 평균 별점': 4,
    '정제 후 별점': 4,
    '별점 변화량': 4,
})

detail_display = demo_store_result[[
    'review_text',
    'rating',
    'photo_count',
    'event_prob',
    'event_pred_name',
    'candidate_2_weight',
]].rename(columns={
    'review_text': '리뷰',
    'rating': '별점',
    'photo_count': '사진 수',
    'event_prob': '이벤트 확률',
    'event_pred_name': '이벤트 예측',
    'candidate_2_weight': '후보 2 가중치',
}).round({
    '이벤트 확률': 4,
    '후보 2 가중치': 4,
})

assert np.all((store_event_prob >= 0) & (store_event_prob <= 1))
np.testing.assert_allclose(store_candidate_2_weight, 1 - store_event_prob)

print('리뷰별 이벤트 확률과 후보 2 가중치')
display(detail_display)
print('가게 단위 후보 2 별점 정제 결과')
display(store_summary)

리뷰별 이벤트 확률과 후보 2 가중치


,리뷰,별점,사진 수,이벤트 확률,이벤트 예측,후보 2 가중치
0,리뷰 이벤트 참여합니다 콜라 부탁드려요,5.0,1,0.4171,일반 리뷰,0.5829
1,양 많고 맛있어요 다음에 또 시킬게요,5.0,0,0.4131,일반 리뷰,0.5869
2,음식이 조금 식어서 왔어요 그래도 괜찮았습니다,3.0,0,0.3113,일반 리뷰,0.6887
3,포토리뷰 약속했어요 서비스 감사합니다,5.0,2,0.2956,일반 리뷰,0.7044
4,배달도 빠르고 포장도 깔끔했어요,4.0,0,0.1834,일반 리뷰,0.8166


가게 단위 후보 2 별점 정제 결과


,리뷰 수,이벤트 예측 수,기존 평균 별점,정제 후 별점,별점 변화량
0,5,0,4.4,4.3508,-0.0492


## 6. 직접 입력 이벤트 리뷰 판별기 (선택 실행)

- 직접 입력을 테스트하려면 `RUN_INTERACTIVE_DEMO = True`로 바꾼 뒤 실행한다.
- 실행하면 리뷰 내용, 사진 수, 별점을 직접 입력받는다.
- 모델은 리뷰 텍스트와 사진 수로 이벤트 리뷰 확률을 계산한다.
- 별점은 모델 입력에 넣지 않고, 후보 2 별점 반영 가중치의 의미를 확인하는 용도로만 함께 보여준다.
- 실제 가게 별점 정제는 5번처럼 여러 리뷰를 모아 가중 평균으로 계산한다.


In [20]:
RUN_INTERACTIVE_DEMO = False

if RUN_INTERACTIVE_DEMO:
    input_review_text = input('리뷰 내용을 입력하세요: ').strip()
    input_photo_text = input('사진 수를 입력하세요 (없으면 0): ').strip()
    input_rating_text = input('별점을 입력하세요 (예: 5.0): ').strip()

    input_photo_count = int(input_photo_text) if input_photo_text else 0
    input_rating = float(input_rating_text) if input_rating_text else np.nan
else:
    input_review_text = '가격도 괜찮은데 맛도 좋고 배달도 넘 빨리와서 좋아요! 잘먹었습니다!'
    input_photo_count = 1
    input_rating = 5.0
    print('RUN_INTERACTIVE_DEMO=False이므로 예시 입력으로 실행합니다.')

if input_review_text == '':
    raise ValueError('리뷰 내용은 비워둘 수 없습니다.')
if input_photo_count < 0:
    raise ValueError('사진 수는 0 이상이어야 합니다.')

input_feature, input_info = make_feature_frame(input_review_text, input_photo_count)
input_proba = event_model.predict_proba(input_feature)[0]
input_general_prob = float(input_proba[0])
input_event_prob = float(input_proba[1])
input_event_pred = int(input_event_prob >= threshold)
input_rating_weight = float(np.clip(1 - input_event_prob, 0, 1))
input_result = pd.DataFrame([{
    '입력 리뷰': input_review_text,
    '정제 리뷰': input_info.loc[0, 'cleaned_review_text'],
    '입력 별점': input_rating,
    '사진 수': input_info.loc[0, 'photo_count'],
    '텍스트 길이': input_info.loc[0, 'text_length'],
    '이모지 수': input_info.loc[0, 'emoji_count'],
    '일반 리뷰 확률': input_general_prob,
    '이벤트 리뷰 확률': input_event_prob,
    'threshold': threshold,
    '최종 판정': '이벤트 리뷰' if input_event_pred == 1 else '일반 리뷰',
    '별점 반영 가중치': input_rating_weight,
}]).round({
    '일반 리뷰 확률': 4,
    '이벤트 리뷰 확률': 4,
    'threshold': 4,
    '별점 반영 가중치': 4,
})

assert 0 <= input_general_prob <= 1
assert 0 <= input_event_prob <= 1
assert np.isclose(input_general_prob + input_event_prob, 1.0)
assert np.isclose(input_rating_weight, 1 - input_event_prob)
assert 'rating' not in feature_cols
display(input_result)


RUN_INTERACTIVE_DEMO=False이므로 예시 입력으로 실행합니다.


,입력 리뷰,정제 리뷰,입력 별점,사진 수,텍스트 길이,이모지 수,일반 리뷰 확률,이벤트 리뷰 확률,threshold,최종 판정,별점 반영 가중치
0,가격도 괜찮은데 맛도 좋고 배달도 넘 빨리와서 좋아요! 잘먹었습니다!,가격도 괜찮은데 맛도 좋고 배달도 넘 빨리와서 좋아요! 잘먹었습니다!,5.0,1,38,0,0.4072,0.5928,0.5009,이벤트 리뷰,0.4072


## 7. CSV 입력 기반 여러 리뷰 판별 및 별점 정제

- `csv/end-to-end-test.csv`를 읽어 여러 리뷰를 한 번에 판별한다.
- 현재 CSV는 총 30개 리뷰로 구성되어 있으며, 기존 전처리 CSV에서 가져온 실제 리뷰 20개와 데모용으로 작성한 리뷰 10개를 포함한다.
- CSV 입력 컬럼은 `review_text`, `rating`, `photo_count`만 사용한다.
- 리뷰별 이벤트 확률과 최종 판정을 출력하고, 후보 2 가중 평균으로 전체 입력 리뷰의 정제 별점을 계산한다.
- 이 결과는 성능 평가가 아니라, 최종 모델을 실제 리뷰 묶음에 적용하는 흐름을 보여주는 예시이다.


In [21]:
CSV_INPUT_PATH = Path('csv/end-to-end-test.csv')

if not CSV_INPUT_PATH.exists():
    raise FileNotFoundError(f'CSV 입력 파일이 없습니다: {CSV_INPUT_PATH}')

csv_input_df = pd.read_csv(CSV_INPUT_PATH)
required_csv_cols = {'review_text', 'rating', 'photo_count'}
missing_csv_cols = required_csv_cols - set(csv_input_df.columns)
if missing_csv_cols:
    raise ValueError(f'CSV 입력에 필요한 컬럼이 없습니다: {sorted(missing_csv_cols)}')
if csv_input_df.empty:
    raise ValueError('CSV 입력 데이터가 비어 있습니다.')

csv_input_df = csv_input_df.copy()
csv_input_df['review_text'] = csv_input_df['review_text'].fillna('').astype(str)
csv_input_df['rating'] = pd.to_numeric(csv_input_df['rating'], errors='coerce')
csv_input_df['photo_count'] = pd.to_numeric(csv_input_df['photo_count'], errors='coerce').fillna(0).astype(int)

if csv_input_df['review_text'].str.strip().eq('').any():
    raise ValueError('review_text가 비어 있는 행이 있습니다.')
if csv_input_df['rating'].isna().any():
    raise ValueError('rating을 숫자로 변환할 수 없는 행이 있습니다.')

csv_feature, csv_info = make_feature_frame(
    csv_input_df['review_text'].tolist(),
    csv_input_df['photo_count'].tolist(),
)
csv_proba = event_model.predict_proba(csv_feature)
csv_general_prob = csv_proba[:, 0]
csv_event_prob = csv_proba[:, 1]
csv_event_pred = (csv_event_prob >= threshold).astype(int)
csv_rating_weight = np.clip(1 - csv_event_prob, 0, 1)
csv_weighted_rating = csv_input_df['rating'].to_numpy() * csv_rating_weight

csv_result = csv_input_df.copy()
csv_result['cleaned_review_text'] = csv_info['cleaned_review_text']
csv_result['general_prob'] = csv_general_prob
csv_result['event_prob'] = csv_event_prob
csv_result['event_pred'] = csv_event_pred
csv_result['event_pred_name'] = np.where(csv_event_pred == 1, '이벤트 리뷰', '일반 리뷰')
csv_result['rating_weight'] = csv_rating_weight
csv_result['weighted_rating'] = csv_weighted_rating

csv_original_mean_rating = float(csv_result['rating'].mean())
csv_weight_sum = float(csv_result['rating_weight'].sum())
csv_refined_rating = (
    csv_original_mean_rating
    if csv_weight_sum <= 0
    else float(csv_result['weighted_rating'].sum() / csv_weight_sum)
)
csv_rating_delta = csv_refined_rating - csv_original_mean_rating

csv_detail_display = csv_result[[
    'review_text',
    'rating',
    'photo_count',
    'general_prob',
    'event_prob',
    'event_pred_name',
    'rating_weight',
]].rename(columns={
    'review_text': '리뷰',
    'rating': '별점',
    'photo_count': '사진 수',
    'general_prob': '일반 리뷰 확률',
    'event_prob': '이벤트 리뷰 확률',
    'event_pred_name': '최종 판정',
    'rating_weight': '별점 반영 가중치',
}).round({
    '일반 리뷰 확률': 4,
    '이벤트 리뷰 확률': 4,
    '별점 반영 가중치': 4,
})

csv_summary = pd.DataFrame([{
    'CSV 경로': str(CSV_INPUT_PATH),
    '입력 리뷰 수': len(csv_result),
    '이벤트 예측 수': int(csv_result['event_pred'].sum()),
    '이벤트 예측 비율': float(csv_result['event_pred'].mean()),
    '기존 평균 별점': csv_original_mean_rating,
    '후보 2 정제 후 별점': csv_refined_rating,
    '별점 변화량': csv_rating_delta,
    '평균 별점 반영 가중치': float(csv_result['rating_weight'].mean()),
}]).round({
    '이벤트 예측 비율': 4,
    '기존 평균 별점': 4,
    '후보 2 정제 후 별점': 4,
    '별점 변화량': 4,
    '평균 별점 반영 가중치': 4,
})

assert len(csv_result) == len(csv_input_df)
assert np.all((csv_general_prob >= 0) & (csv_general_prob <= 1))
assert np.all((csv_event_prob >= 0) & (csv_event_prob <= 1))
np.testing.assert_allclose(csv_general_prob + csv_event_prob, 1.0)
np.testing.assert_allclose(csv_rating_weight, 1 - csv_event_prob)
if csv_weight_sum > 0:
    expected_csv_refined_rating = float((csv_result['rating'] * csv_result['rating_weight']).sum() / csv_weight_sum)
    assert np.isclose(csv_refined_rating, expected_csv_refined_rating)
assert 'rating' not in feature_cols

print('CSV 입력 리뷰별 이벤트 판정')
display(csv_detail_display)
print('CSV 입력 기반 후보 2 별점 정제 요약')
display(csv_summary)


CSV 입력 리뷰별 이벤트 판정


,리뷰,별점,사진 수,일반 리뷰 확률,이벤트 리뷰 확률,최종 판정,별점 반영 가중치
0,리뷰만 보고 시켰어요 다들 칭찬하는 이유가 있었네요\r\n덜 맴게 시켰는데 처음...,5.0,1,0.6402,0.3598,일반 리뷰,0.6402
1,거짓말 안하고 먹고 맛있어서 바로 또 주문 했어요 자주 먹을꺼같아요 진짜 최고에요ㅠ...,5.0,1,0.4136,0.5864,이벤트 리뷰,0.4136
2,너무 맛있는학돌이네..🤭💛❤ 매운맛으로 주먹밥이 훌훌들어가요 담에 또뵈용,5.0,1,0.6360,0.3640,일반 리뷰,0.6360
3,11월에 여기 처음 알고\r\n벌써 3번째 주문입니다리\r\n쿠ㅍ2번 배민은 오늘처...,5.0,4,0.4154,0.5846,이벤트 리뷰,0.4154
4,맛있다는 리뷰 염탐만하다 처음 시켜봤는데 맛있어요ㅜㅜ 게 땡길때 비릴지 걱정하며 게...,5.0,1,0.3673,0.6327,이벤트 리뷰,0.3673
5,논현 본점에서 유명한 학돌이네 꽃게무침!🦀\r\n인스타에서 자주 보여서 꼭 한번 먹...,3.0,1,0.4274,0.5726,이벤트 리뷰,0.4274
6,뒷끝없는 깔끔한 매운맛이라 좋아요! \r\n양념게장이 아닌 진짜 무침인데 넘 신선하...,5.0,0,0.5343,0.4657,일반 리뷰,0.5343
7,"와우 진짜 핵존맛탱이에요!\r\n꽃게 무침은 처음먹어보는데, 완전 시원~~ 하고 비...",5.0,2,0.3484,0.6516,이벤트 리뷰,0.3484
8,이번달만 벌써 네번째예요 진짜 매일 먹구 싶은 학돌이 🦀🦀🦀🦀🦀\r\n게 살도 많고...,5.0,1,0.2823,0.7177,이벤트 리뷰,0.2823
9,진짜 너무너무너무 맛있어요!! 그리고 살도 많고 비린맛에 예민한데 비린맛 0.000...,5.0,1,0.5102,0.4898,일반 리뷰,0.5102


CSV 입력 기반 후보 2 별점 정제 요약


,CSV 경로,입력 리뷰 수,이벤트 예측 수,이벤트 예측 비율,기존 평균 별점,후보 2 정제 후 별점,별점 변화량,평균 별점 반영 가중치
0,csv\end-to-end-test.csv,30,12,0.4,4.8,4.7542,-0.0458,0.5696


## 8. 결과 해석

- 이벤트 확률이 높은 리뷰는 후보 2 가중치가 낮아져 별점 평균에 덜 반영된다.
- 이벤트 확률이 낮은 리뷰는 일반 리뷰에 가깝다고 보고 별점 영향력을 더 유지한다.
- 직접 입력 판별기는 같은 모델을 사용해 사용자가 입력한 리뷰의 이벤트 확률과 별점 반영 가중치를 확인하는 용도이다.
- CSV 입력 데모는 여러 리뷰를 묶어 가게 단위 별점 정제 흐름을 확인하는 적용 예시이다.
- 이 노트북은 최종 모델 적용 데모이며, 모델 학습과 성능 비교는 04~07번에서 수행한다.
